# Testing Performance Metrics For The Agent

In [1]:
import pandas as pd

In [2]:
df = pd.read_json("sms_conversations.json")

In [3]:
df

,conversation_id,candidate_phone,recruiter_phone,start_time_utc,turns
0,1,+1-555-0201,+1-555-0000,2026-04-03T15:12:00Z,"[{'turn_id': 1, 'speaker': 'recruiter', 'times..."
1,2,+1-555-0202,+1-555-0000,2026-04-30T11:19:00Z,"[{'turn_id': 1, 'speaker': 'recruiter', 'times..."
2,3,+1-555-0203,+1-555-0000,2026-04-10T16:02:00Z,"[{'turn_id': 1, 'speaker': 'recruiter', 'times..."
3,4,+1-555-0204,+1-555-0000,2026-04-28T11:13:00Z,"[{'turn_id': 1, 'speaker': 'recruiter', 'times..."
4,5,+1-555-0205,+1-555-0000,2026-04-26T12:43:00Z,"[{'turn_id': 1, 'speaker': 'recruiter', 'times..."
5,6,+1-555-0206,+1-555-0000,2026-04-16T16:45:00Z,"[{'turn_id': 1, 'speaker': 'recruiter', 'times..."
6,7,+1-555-0207,+1-555-0000,2026-04-12T12:15:00Z,"[{'turn_id': 1, 'speaker': 'recruiter', 'times..."
7,8,+1-555-0208,+1-555-0000,2026-04-03T12:49:00Z,"[{'turn_id': 1, 'speaker': 'recruiter', 'times..."
8,9,+1-555-0209,+1-555-0000,2026-04-17T17:20:00Z,"[{'turn_id': 1, 'speaker': 'recruiter', 'times..."
9,10,+1-555-0210,+1-555-0000,2026-04-27T13:30:00Z,"[{'turn_id': 1, 'speaker': 'recruiter', 'times..."


## Sum up

<p>
We got some conversations to test on: the file contains <i>convesation id</i> which is equivalent to <i>Session id</i> in our code, some <i>meta data</i> on the conversation, and the <i>turns</i> in each conversation.
</p>
<p>
We need to create a turn based test bench, which takes the turns in each conversation and checks if the agent is following the script as he should or not.
</p>



### The Evaluation Process is as follows: 
1. Load the conversations - already done up there
2. Extract the label from each turn in each conversation.
3. Extract the initial prompt from each conversation.
4. Extract the user's prompts from each conversation.
5. Start the Agent with the initial response.
6. Send the user's prompt to the Agent in each turn.
7. Get the agent's response and intention for each user prompt.
8. Evaluate the agent's intention against the extracted label.
9. Save the agent's intentions into an array.
10. Create a confusion matrix for each label ('continue', 'schedule', 'exit')


In [ ]:
# all these are needed after we create the evaluation platform.

import os
from dotenv import load_dotenv

os.chdir('../')
os.getcwd()

from langchain.tools import tool
from app.modules.agents.agents import Agent
from app.modules.embedding import build_search_job_description_tool, Embedder

load_dotenv()

# ollama chat llms: gemma4:e4b

# Create an Embedding db if not found:
if 'chroma_db' not in os.listdir():
    embed = Embedder()

    embed.build_vectorstore('../Python Developer Job Description.pdf')

# Create the info tool function
load_info = build_search_job_description_tool()

# Import and add the scheduling tools here


In [6]:
for i,r in df.iterrows():
    print(r)

conversation_id                                                    1
candidate_phone                                          +1-555-0201
recruiter_phone                                          +1-555-0000
start_time_utc                                  2026-04-03T15:12:00Z
turns              [{'turn_id': 1, 'speaker': 'recruiter', 'times...
Name: 0, dtype: object
conversation_id                                                    2
candidate_phone                                          +1-555-0202
recruiter_phone                                          +1-555-0000
start_time_utc                                  2026-04-30T11:19:00Z
turns              [{'turn_id': 1, 'speaker': 'recruiter', 'times...
Name: 1, dtype: object
conversation_id                                                    3
candidate_phone                                          +1-555-0203
recruiter_phone                                          +1-555-0000
start_time_utc                                  2026-04-1

In [ ]:
for i,r in df.iterrows():
    user_message = None
    message = r['turns'][0]['text']
    @tool
    def get_conversation_time()-> str:
        '''Returns the start time of the conversation'''
        return r['start_time_utc']

    sys_msg =f""""The User's application has been received successfully. The User will be redirected to you."
    Output exactly this: {message},
    to the user **at the start of the conversation**, After consulting the advisor, Ignoring the Advisor's output, and after that you can continue normally."""

    agent = Agent(
        api_key="ollama",
        model="gemma4:e4b", 
        base_url="http://localhost:11434/v1",
        system_message=sys_msg,
        sch_tools=[get_conversation_time], # add the scheduling tools here
        info_tools=[load_info],
        temperature = 0,
        verbose=True
    )

    for turn in r['turns']:
        print('-'*60)
        if turn['turn_id'] == 1:
            ai_mesasge = agent.step('candidate')
            print(ai_mesasge)
        if turn['speaker'] == 'candidate':
            print(turn['text'])
            ai_mesasge = agent.step('candidate', turn['text'])
            print(ai_mesasge)
        elif turn['speaker'] == 'recruiter':
            print('Recruiter Label:')
            print(turn['label'])
            print('Assistant Label:')
            try:
                print(agent.get_from_store('candidate', False)[-1])
            except IndexError:
                print('continue')

    print('-'*60)
    print('Agent Intentions in the conversation:')
    print(agent.get_from_store('candidate', False))
    print('-'*60)
    print('Summaries in conversation:')
    print(agent.summary_counter)
